# Cell 1: Environment Setup and Dependencies

This first step is crucial. We will install all the necessary Python libraries for our project. To avoid dependency conflicts, which are common in complex AI environments, we are installing **specific versions** of each package that are known to work well together. This is a best practice called 'pinning dependencies'.

In [ ]:
# ====================================================================
# MENOPAUSE AI EDUCATION SYSTEM - AGENTIC IMPLEMENTATION
# Breaking the Silence: Multi-Agent Culturally-Sensitive AI System
# ====================================================================

# === Updated Installation Cell ===
import sys
import subprocess

def install_packages():
    packages = [
        "pandas==2.1.4",
        "scikit-learn==1.3.2",
        "chromadb==0.4.24",
        "sentence-transformers==2.2.2",
        "torch==2.1.0",
        "transformers==4.36.2",
        "matplotlib==3.8.2",
        "seaborn==0.13.0",
        "plotly==5.18.0",
        "python-dotenv==1.0.0"
    ]
    print("Installing required packages...")
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])
            print(f"  [✓] Successfully installed {package}")
        except subprocess.CalledProcessError as e:
            print(f"  [✗] Failed to install {package}: {e}")

# Run the installation
install_packages()

# === Verified Imports ===
import pandas as pd
import re
from typing import Dict, List, Any
import chromadb
from chromadb.utils import embedding_functions
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from dotenv import load_dotenv

print("\n✅ All dependencies are ready!")

# Import core libraries
# import os
# import json
# import numpy as np
# import pandas as pd
# from datetime import datetime
# from typing import Dict, List, Optional, Tuple, Any
# import asyncio
# from dotenv import load_dotenv

load_dotenv()

# Camel AI imports for multi-agent systems
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from camel.agents import ChatAgent
from camel.messages import BaseMessage

# ChromaDB imports for vector storage
import chromadb
from chromadb.utils import embedding_functions

# Hugging Face imports
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

print("\n🚀 Menopause AI Education System - Agentic Architecture Ready!")
print("🔧 Using: Camel AI + ChromaDB + Hugging Face Transformers")

### A Note on Dependency Conflicts

You might have previously seen an error message like `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed`. This is a very common issue in Python development.

**What does it mean?**
This online environment already has many packages pre-installed. When we try to install new packages (like `camel-ai` or `transformers`), `pip` (Python's package installer) tries to find versions that are compatible with *everything* that's already here. If `camel-ai` needs one version of a library but a pre-installed package needs a different version, a conflict occurs.

**How We Solved It:**
In the cell above, instead of just asking for the latest version of a package (e.g., `pip install camel-ai`), we provided a list of packages with **specific version numbers** (e.g., `'camel-ai[all]==0.3.0'`). This is called **"pinning dependencies"** and is a professional best practice for creating stable, reproducible AI systems. It tells `pip` *exactly* which versions to install, which resolves the conflicts.

# Cell 2: ChromaDB Knowledge Base Setup

This component acts as the system's long-term memory. We use `ChromaDB` to store and retrieve specialized medical and cultural information related to menopause. This allows our AI to provide responses grounded in facts, a technique known as Retrieval-Augmented Generation (RAG).

In [ ]:
# ====================================================================
# CHROMADB VECTOR DATABASE SETUP
# ====================================================================

class ChromaDBKnowledgeBase:
    def __init__(self, persist_directory="./menopause_chromadb"):
        """Initialize ChromaDB with persistent storage."""
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        
        self.collections = {}
        self._initialize_collections()
        
        print(f"✅ ChromaDB initialized. Collections: {[c.name for c in self.client.list_collections()]}")
    
    def _initialize_collections(self):
        """Initialize all necessary knowledge collections."""
        collection_names = ["medical_knowledge", "cultural_contexts", "safety_guidelines"]
        for name in collection_names:
            self.collections[name] = self.client.get_or_create_collection(
                name=name, embedding_function=self.embedding_function
            )

    def add_knowledge(self, collection_name: str, documents: List[str], ids: List[str]):
        if self.collections[collection_name].count() > 0: return
        self.collections[collection_name].add(documents=documents, ids=ids)
        print(f"  -> Populated '{collection_name}' with {len(documents)} documents.")

    def query_knowledge(self, collection_name: str, query: str, n_results: int = 3) -> Dict:
        return self.collections[collection_name].query(query_texts=[query], n_results=n_results)

    def populate_initial_knowledge(self):
        """Populates all collections with initial data."""
        print("\n🌍 Populating Initial Knowledge Base...")
        medical_docs = [
            "Menopause typically occurs between ages 45-55.",
            "Hot flashes, the most common symptom, are caused by hormonal fluctuations affecting the body's thermostat.",
            "Hormone Replacement Therapy (HRT) can manage severe symptoms but carries risks and requires medical consultation.",
            "Lower estrogen after menopause increases the risk of osteoporosis."
        ]
        self.add_knowledge("medical_knowledge", medical_docs, [f"med_{i}" for i in range(len(medical_docs))])

        cultural_docs = [
            "In some Indian cultures, menopause (rajonivritti) is seen as a liberation from menstrual restrictions.",
            "Western cultures often frame menopause as a medical condition focusing on the loss of youth and fertility.",
            "In Japan, 'konenki' is viewed as a natural life transition, not just a medical event."
        ]
        self.add_knowledge("cultural_contexts", cultural_docs, [f"cult_{i}" for i in range(len(cultural_docs))])
        print("Knowledge Base is ready.")

# Initialize and populate the knowledge base
knowledge_base = ChromaDBKnowledgeBase()
knowledge_base.populate_initial_knowledge()

# Test query
test_results = knowledge_base.query_knowledge("cultural_contexts", "Indian women menopause liberation")
print(f"\n🔍 Test Query Results: Found {len(test_results['documents'][0])} relevant document(s).")
print(f"  -> Most relevant doc: {test_results['documents'][0][0]}")

# Cell 3: Camel AI Multi-Agent System

Here we define the 'mind' of our system using the `camel-ai` framework. We create a society of specialized AI agents, each with a specific role. This division of labor makes the system more robust, transparent, and easier to manage.

-   **Cultural Expert**: Assesses cultural context and adapts language.
-   **Medical Expert**: Ensures information is medically accurate.
-   **Safety Validator**: Checks for bias and harmful content.
-   **Coordinator**: Manages the other agents and synthesizes the final response.

**Note**: This cell requires an OpenAI API key. To run it, you must have a file named `.env` in the same directory with the content: `OPENAI_API_KEY='your_key_here'`. If you don't have a key, the code will gracefully fall back to a simulation.

In [ ]:
# ====================================================================
# CAMEL AI MULTI-AGENT SYSTEM
# ====================================================================

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

class MenopauseAIAgentSociety:
    def __init__(self, knowledge_base: ChromaDBKnowledgeBase):
        self.knowledge_base = knowledge_base
        self.agents = {}
        self.is_simulation = not OPENAI_API_KEY

        if self.is_simulation:
            print("⚠️ OpenAI API key not found. Running in simulation mode.")
        else:
            print("✅ OpenAI API key found. Initializing real agents.")
            # Correct way to create a model instance in camel-ai v0.3.0
            self.model = ModelFactory.create(
                model_platform=ModelPlatformType.OPENAI,
                model_type=ModelType.GPT_4O_MINI,
                model_config_dict={"api_key": OPENAI_API_KEY, "temperature": 0.2}
            )
        self._create_agents()
        print("🤖 Multi-Agent Society initialized")

    def _create_agents(self):
        agent_definitions = {
            'cultural_expert': "You are a Cultural Expert on menopause. Your job is to analyze user queries to determine their cultural perspective (liberation, medical, or neutral) and adapt information to be culturally sensitive.",
            'medical_expert': "You are a Medical Expert on menopause. Your job is to provide and verify medically accurate, evidence-based information using the context provided.",
            'safety_validator': "You are a Safety Validator. Your job is to scan responses for any bias, harmful advice, or unsafe language.",
            'coordinator': "You are a Content Coordinator. Your job is to synthesize inputs from all experts into a single, coherent, and helpful response for the user."
        }
        for name, sys_prompt in agent_definitions.items():
            if self.is_simulation:
                # Create a mock agent for simulation mode
                self.agents[name] = lambda p: f"(Simulated {name.replace('_', ' ').title()})\nPrompt: {p[:100]}..."
            else:
                system_message = BaseMessage.make_assistant_message(role_name=name, content=sys_prompt)
                self.agents[name] = ChatAgent(model=self.model, system_message=system_message)
        print(f"  -> Created {len(self.agents)} agents.")

    async def _execute_step(self, agent_name: str, prompt: str) -> str:
        if self.is_simulation:
            await asyncio.sleep(0.1) # Simulate async call
            return self.agents[agent_name](prompt)
        else:
            response = await self.agents[agent_name].step(prompt)
            if response:
                return response.msgs[0].content
            return "Error: No response from agent."

    async def process_user_query(self, user_query: str, pathway: str) -> Dict[str, Any]:
        print(f"\n🔄 Processing query via multi-agent collaboration (Pathway: {pathway.upper()})...")
        
        # 1. Retrieve knowledge from ChromaDB
        medical_context = self.knowledge_base.query_knowledge("medical_knowledge", user_query)
        cultural_context = self.knowledge_base.query_knowledge("cultural_contexts", user_query)
        context_str = f"Medical Context: {medical_context}\nCultural Context: {cultural_context}"
        
        # 2. Medical Expert generates the core medical information
        medical_prompt = f"Using the following context, answer the user's query medically.\nContext: {context_str}\nQuery: {user_query}"
        medical_guidance = await self._execute_step('medical_expert', medical_prompt)
        print("  -> [1/4] Medical Expert provided guidance.")

        # 3. Cultural Expert adapts the medical info
        cultural_prompt = f"Adapt this medical guidance for a user with a '{pathway}' perspective on menopause. Guidance: {medical_guidance}"
        cultural_response = await self._execute_step('cultural_expert', cultural_prompt)
        print("  -> [2/4] Cultural Expert adapted the response.")

        # 4. Safety Validator checks the culturally-adapted response
        safety_prompt = f"Is this response safe, unbiased, and medically responsible? Response: {cultural_response}"
        safety_validation = await self._execute_step('safety_validator', safety_prompt)
        print("  -> [3/4] Safety Validator reviewed the content.")

        # 5. Coordinator synthesizes the final response
        final_prompt = f"Combine the following into a single, helpful response. Original Query: {user_query}, Adapted Guidance: {cultural_response}, Safety Review: {safety_validation}"
        final_response = await self._execute_step('coordinator', final_prompt)
        print("  -> [4/4] Coordinator synthesized the final answer.")

        return {
            "final_response": final_response,
            "medical_guidance": medical_guidance,
            "cultural_response": cultural_response,
            "safety_validation": safety_validation
        }

# Initialize the agent society
agent_society = MenopauseAIAgentSociety(knowledge_base)

# Cell 4: Integrated System Pipeline & Simulation

This is the final orchestration. We create a master class, `MenopauseAISystem`, that manages the entire user interaction from start to finish. It uses all the components we've built: the `ChromaDBKnowledgeBase` for information, the `MenopauseAIAgentSociety` for reasoning, and a local `SafetyValidator` for a final check.

In [ ]:
# ====================================================================
# INTEGRATED SYSTEM PIPELINE
# ====================================================================

class MenopauseAISystem:
    def __init__(self, kb: ChromaDBKnowledgeBase, society: MenopauseAIAgentSociety):
        self.kb = kb
        self.society = society
        self.assessment_module = CulturalAssessmentModule()
        self.router = ContentRouter()
        self.validator = SafetyValidator() # A final, local check
        print("\n🚀 Menopause AI System Orchestrator is Live!")

    async def run_interaction(self, user_query: str, assessment_answers: Dict[str, str]) -> Dict[str, Any]:
        print(f"\n--- Starting New Interaction: '{user_query}' ---")
        # 1. Assess & Route
        scores = self.assessment_module.get_scores_from_answers(assessment_answers)
        pathway = self.router.get_dominant_pathway(scores)
        print(f"[1] Assessment Complete. Determined Pathway: {pathway.upper()}")

        # 2. Process with Agent Society
        agent_results = await self.society.process_user_query(user_query, pathway)
        print("[2] Multi-agent collaboration complete.")

        # 3. Final Local Safety Validation
        final_response_from_agents = agent_results['final_response']
        validation = self.validator.validate_response(final_response_from_agents)
        print(f"[3] Final local safety validation. Approved: {validation['approved']}")

        # 4. Prepare final output
        if validation['approved']:
            final_response = final_response_from_agents + "\n\n*Disclaimer: This is AI-generated educational content. Always consult a healthcare professional for medical advice.*"
        else:
            final_response = f"I understand you have questions. For your safety, a detailed response could not be generated. Issues found: {validation['issues']}. Please consult a healthcare professional."
        
        agent_results['final_response'] = final_response
        agent_results['pathway'] = pathway
        return agent_results

async def main():
    ai_system = MenopauseAISystem(knowledge_base, agent_society)
    
    personas = {
        "Priya (Liberation Pathway)": {
            "query": "I feel happy to be free from monthly cycles. What health aspects should I focus on now?",
            "answers": {'q1': 'A', 'q2': 'A', 'q3': 'A'}
        },
        "Susan (Medical Pathway)": {
            "query": "What are the medical treatments for severe night sweats?",
            "answers": {'q1': 'B', 'q2': 'B', 'q3': 'B'}
        }
    }
    
    session_results = []
    for name, data in personas.items():
        print(f"\n{'='*20} Simulating for: {name} {'='*20}")
        result = await ai_system.run_interaction(data['query'], data['answers'])
        print(f"\n--- FINAL RESPONSE for {name} ---\n{result['final_response']}")
        session_results.append({"persona": name, "pathway": result['pathway']})

    # Visualize results
    results_df = pd.DataFrame(session_results)
    sns.countplot(x='pathway', data=results_df, palette='viridis').set_title('Cultural Pathways Used in Simulation')
    plt.show()

# Run the main asynchronous function
# In a Jupyter Notebook, you can run an async function like this:
import asyncio
asyncio.run(main())